<a href="https://colab.research.google.com/github/2303A52060/High-performace-computing/blob/main/Ass_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AssignmentNumber:11

MPI Point-to-Point: Non-Blocking Communication -Overlap
Assignment 1: Non-Blocking Send and Receive with Overlap
Scenario:

In a distributed system, a master

process must transfer an integer array to

a worker process while both processes

continue performing independent

computations to reduce idle time Objective

To implement non-blocking point-to-point

communication using Isend

and Irecv and demonstrate overlap of

computation with communication.

In [1]:
!pip install mpi4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.2 MB/s eta 0:00:00


In [4]:
!pip install mpi4py
from mpi4py import MPI
import numpy as np
import time

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

if size != 2:
    if rank == 0:
        print("This program requires exactly 2 processes. Please run with `mpiexec -n 2 python your_script_name.py` for full functionality.")
    print(f"Process {rank}: Running in a single-process environment. Skipping MPI communication.")
else:
    tag = 0

    if rank == 0:
        # Process 0 (Master)
        data_to_send = np.arange(10, dtype=np.int32)
        print(f"Process {rank}: Sending data: {data_to_send}")

        # Non-blocking send
        req = comm.Isend([data_to_send, MPI.INT], dest=1, tag=tag)

        # Perform some computation while communication is in progress
        print(f"Process {rank}: Performing computation while sending...")
        for i in range(5):
            time.sleep(0.1) # Simulate computation
            print(f"Process {rank}: Computation step {i+1}")

        # Wait for the send to complete
        req.Wait()
        print(f"Process {rank}: Send completed.")

    elif rank == 1:
        # Process 1 (Worker)
        # Prepare a buffer for receiving data
        data_received = np.empty(10, dtype=np.int32)

        # Non-blocking receive
        req = comm.Irecv([data_received, MPI.INT], source=0, tag=tag)

        # Perform some computation while communication is in progress
        print(f"Process {rank}: Performing computation while receiving...")
        for i in range(5):
            time.sleep(0.15) # Simulate computation
            print(f"Process {rank}: Computation step {i+1}")

        # Wait for the receive to complete
        req.Wait()
        print(f"Process {rank}: Receive completed.")
        print(f"Process {rank}: Received data: {data_received}")

print(f"Process {rank}: Program finished.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.7 MB/s eta 0:00:00
This program requires exactly 2 processes. Please run with `mpiexec -n 2 python your_script_name.py` for full functionality.
Process 0: Running in a single-process environment. Skipping MPI communication.
Process 0: Program finished.


Assignment 2: Overlapping Neighbor Communication with
Computation

Scenario

In a parallel data-processing

application, each process exchanges data

with its neighboring process while

continuing local computation to

improve performance.

Objective

To study non-blocking communication

between neighboring processes

and overlap it with computation.

Tasks

1. Assign local data to each process.

2. Send data to a neighboring process

using Isend.

3. Receive data using Irecv.

4. Compute the sum of local data during communication.

5. Synchronize using Wait or Waitall.
Learning Outcomes

 Understanding neighbor-based communication

 Improved parallel efficiency through overlap


In [2]:
!pip install mpi4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 71.2 MB/s eta 0:00:00


In [3]:
import sys
!{sys.executable} -m pip install mpi4py

from mpi4py import MPI
import numpy as np
import time

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# Ensure at least 2 processes for neighbor communication
if size < 2:
    if rank == 0:
        print("This program requires at least 2 processes for neighbor communication. Please run with `mpiexec -n N python your_script_name.py` where N >= 2.")
    exit()

# 1. Assign local data to each process.
local_data_size = 5
local_data = np.arange(rank * local_data_size, (rank + 1) * local_data_size, dtype=np.int32)
print(f"Process {rank}: Initial local data: {local_data}")

# Determine neighbors (circular communication)
left_neighbor = (rank - 1 + size) % size
right_neighbor = (rank + 1) % size

tag_send = 100
tag_recv = 101

# Prepare buffers for received data
received_from_left = np.empty(local_data_size, dtype=np.int32)

# List to store non-blocking requests
requests = []

# 2. Send data to a neighboring process using Isend.
# Send local_data to the right neighbor
print(f"Process {rank}: Initiating Isend to {right_neighbor}")
req_send = comm.Isend([local_data, MPI.INT], dest=right_neighbor, tag=tag_send)
requests.append(req_send)

# 3. Receive data using Irecv.
# Receive data from the left neighbor
print(f"Process {rank}: Initiating Irecv from {left_neighbor}")
req_recv = comm.Irecv([received_from_left, MPI.INT], source=left_neighbor, tag=tag_send) # Note: source uses send_tag
requests.append(req_recv)

# 4. Compute the sum of local data during communication.
print(f"Process {rank}: Performing local computation while communication is in progress...")
local_sum_during_comm = 0
for x in local_data:
    local_sum_during_comm += x
    time.sleep(0.05) # Simulate computation
print(f"Process {rank}: Local computation (sum of initial data) during communication: {local_sum_during_comm}")

# 5. Synchronize using Wait or Waitall.
# Wait for all non-blocking communication requests to complete
MPI.Request.Waitall(requests)
print(f"Process {rank}: All communication requests completed.")

print(f"Process {rank}: Received data from {left_neighbor}: {received_from_left}")

print(f"Process {rank}: Program finished.")


This program requires at least 2 processes for neighbor communication. Please run with `mpiexec -n N python your_script_name.py` where N >= 2.
Process 0: Initial local data: [0 1 2 3 4]
Process 0: Initiating Isend to 0
Process 0: Initiating Irecv from 0
Process 0: Performing local computation while communication is in progress...
Process 0: Local computation (sum of initial data) during communication: 10
Process 0: All communication requests completed.
Process 0: Received data from 0: [0 1 2 3 4]
Process 0: Program finished.


Assignment 3: Non-Blocking Receive Using Test

Scenario

In real-time parallel applications,

processes must continue useful

computation while periodically checking

whether incoming data

has arrived.

Objective

To use the Test operation in MPI for

monitoring non-blocking

communication completion.

Tasks

1. Initiate a non-blocking receive using

Irecv.

2. Perform computation in a loop.

3. Periodically check communication

status using Test.

4. Process the received data once
communication completes.

Learning Outcomes

 Understanding asynchronous MPI communication


 Effective use of Test for progress checking


In [1]:
import sys
!{sys.executable} -m pip install mpi4py

from mpi4py import MPI
import numpy as np
import time

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

if size != 2:
    if rank == 0:
        print("This program requires exactly 2 processes. Please run with `mpiexec -n 2 python your_script_name.py` for full functionality.")
    print(f"Process {rank}: Running in a single-process environment. Skipping MPI communication.")
else:
    tag = 0
    data_size = 10

    if rank == 0:
        # Process 0 (Master) - Sends data
        data_to_send = np.arange(data_size, dtype=np.int32)
        print(f"Process {rank}: Sending data: {data_to_send}")
        comm.Send([data_to_send, MPI.INT], dest=1, tag=tag)
        print(f"Process {rank}: Send completed.")

    elif rank == 1:
        # Process 1 (Worker) - Non-blocking receive with Test
        data_received = np.empty(data_size, dtype=np.int32)

        # 1. Initiate a non-blocking receive using Irecv.
        print(f"Process {rank}: Initiating Irecv from source 0.")
        req = comm.Irecv([data_received, MPI.INT], source=0, tag=tag)

        # 2. Perform computation in a loop.
        # 3. Periodically check communication status using Test.
        done = False
        computation_steps = 0
        print(f"Process {rank}: Performing computation while waiting for data...")
        while not done:
            # Simulate computation
            time.sleep(0.05)
            computation_steps += 1
            print(f"Process {rank}: Computation step {computation_steps}")

            # Check if receive is complete
            done, status = req.Test()

            if done:
                print(f"Process {rank}: Receive completed after {computation_steps} computation steps.")
                # 4. Process the received data once communication completes.
                print(f"Process {rank}: Received data: {data_received}")
            else:
                # Optional: Add a timeout or limit for computation steps to prevent infinite loop
                if computation_steps > 100: # Example limit
                    print(f"Process {rank}: Computation limit reached before receive completion. Exiting.")
                    break

print(f"Process {rank}: Program finished.")


This program requires exactly 2 processes. Please run with `mpiexec -n 2 python your_script_name.py` for full functionality.
Process 0: Running in a single-process environment. Skipping MPI communication.
Process 0: Program finished.
